# <b>Preprocessing Data from InfluxDB</b>

In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
import influxdb_client, os, time
from influxdb_client.client.write_api import SYNCHRONOUS
from influxdb_client import InfluxDBClient, Point, WritePrecision

In [2]:
# Configuration
load_dotenv()

TOKEN = os.environ.get("INFLUXDB_TOKEN")
URL = os.environ.get("INFLUXDB_URL")
ORG = os.environ.get("INFLUXDB_ORG")
BUCKET = os.environ.get("INFLUXDB_BUCKET")

missing = [
    name
    for name, value in {
        "INFLUXDB_TOKEN": TOKEN,
        "INFLUXDB_URL": URL,
        "INFLUXDB_ORG": ORG,
        "INFLUXDB_BUCKET": BUCKET,
    }.items()
    if not value
]

if missing:
    raise ValueError(f"Missing required environment variables: {', '.join(missing)}")

In [3]:
write_client = influxdb_client.InfluxDBClient(url=URL, token=TOKEN, org=ORG)

In [4]:
bucket="2.2_BUCKET_TEST"

write_api = write_client.write_api(write_options=SYNCHRONOUS)
   
for value in range(5):
  point = (
    Point("measurement1")
    .tag("tagname1", "tagvalue1")
    .field("field1", value)
  )
  write_api.write(bucket=bucket, org="sl2", record=point)
  time.sleep(1) # separate points by 1 second

In [5]:
bucket="2.2_BUCKET_TEST"

write_api = write_client.write_api(write_options=SYNCHRONOUS)
   
for value in range(5):
  point = (
    Point("measurement1")
    .tag("tagname1", "tagvalue1")
    .field("field1", value)
  )
  write_api.write(bucket=bucket, org="sl2", record=point)
  time.sleep(1) # separate points by 1 second

In [6]:
query_api = write_client.query_api()

query = """from(bucket: "2.2_BUCKET_TEST")
  |> range(start: -10m)
  |> filter(fn: (r) => r._measurement == "measurement1")
  |> mean()"""
tables = query_api.query(query, org="sl2")

for table in tables:
    for record in table.records:
        print(record)


FluxRecord() table: 0, {'result': '_result', 'table': 0, '_start': datetime.datetime(2026, 5, 18, 6, 35, 34, 753186, tzinfo=datetime.timezone.utc), '_stop': datetime.datetime(2026, 5, 18, 6, 45, 34, 753186, tzinfo=datetime.timezone.utc), '_value': 2.0, '_field': 'field1', '_measurement': 'measurement1', 'tagname1': 'tagvalue1'}
